# 03 - Baseline Augmentation + RuleFilter

**Objectif**: comparer proprement les baselines TF-IDF avec et sans augmentation de sécurité.

Ce notebook garde les mêmes artefacts que la version précédente, mais sépare clairement:
1. génération des exemples augmentés,
2. création du train augmenté,
3. entraînement des modèles augmentés,
4. évaluation et analyse.

**Important**: `val.csv`, `test.csv` et `dataset_master.csv` ne sont pas modifiés.

**Systèmes évalués**:
- `original_logreg`
- `original_linear_svm`
- `augmented_logreg_rulefilter`
- `augmented_linear_svm_rulefilter`

**Lecture des résultats**:
- `original_linear_svm` reste la meilleure baseline automatique.
- `augmented_linear_svm_rulefilter` vise surtout un meilleur comportement manuel/sécurité.
- Le système augmenté ne remplace donc pas automatiquement la baseline originale; il répond à un objectif de sécurité différent.


## 1. Setup et configuration

Le notebook orchestre l'expérience. Les fonctions réutilisables vivent dans `src/evaluation/baseline_augmented_utils.py`.

In [ ]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'data').exists() and (cwd / 'models').exists():
        return cwd
    if cwd.name == 'notebooks' and (cwd.parent / 'data').exists():
        return cwd.parent
    raise RuntimeError('Cannot locate project root')


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.baseline_augmented_utils import (  # noqa: E402
    EMERGENCY_PATTERNS,
    LABEL_ORDER,
    OOS_PATTERNS,
    best_system_for_split,
    build_manual_augmentation,
    build_manual_safety_tests,
    build_train_augmented,
    comparison_table,
    evaluate_systems,
    file_md5,
    final_readiness,
    load_split,
    make_augmented_models,
    save_augmented_model,
    save_evaluation_outputs,
    split_label_distribution,
)

DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_AUG = ROOT / 'data' / 'augmentation'
MODELS_DIR = ROOT / 'models'
REPORTS_DIR = ROOT / 'reports' / 'baselines'

TRAIN_PATH = DATA_PROCESSED / 'train.csv'
VAL_PATH = DATA_PROCESSED / 'val.csv'
TEST_PATH = DATA_PROCESSED / 'test.csv'
TRAIN_AUG_PATH = DATA_PROCESSED / 'train_augmented.csv'
MANUAL_AUG_PATH = DATA_AUG / 'manual_train_examples.csv'
MANUAL_TEST_PATH = DATA_AUG / 'manual_safety_tests.csv'

ORIG_LOGREG_DIR = MODELS_DIR / 'baseline_tfidf_logreg'
ORIG_SVM_DIR = MODELS_DIR / 'baseline_tfidf_linear_svm'
AUG_LOGREG_DIR = MODELS_DIR / 'baseline_tfidf_logreg_augmented'
AUG_SVM_DIR = MODELS_DIR / 'baseline_tfidf_linear_svm_augmented'

for directory in [DATA_AUG, REPORTS_DIR, AUG_LOGREG_DIR, AUG_SVM_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Configuration complete')
print(f'ROOT: {ROOT}')
print(f'Reports: {REPORTS_DIR}')


Configuration complete
ROOT: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot
Reports: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\reports\baselines


## 2. Chargement et validation des splits

On calcule les hash avant l'expérience pour vérifier en fin de notebook que les splits d'origine n'ont pas été modifiés.

In [ ]:
train_hash_before = file_md5(TRAIN_PATH)
val_hash_before = file_md5(VAL_PATH)
test_hash_before = file_md5(TEST_PATH)

train_df = load_split(TRAIN_PATH, 'train')
val_df = load_split(VAL_PATH, 'val')
test_df = load_split(TEST_PATH, 'test')

print('Données chargées')
print(f'train: {len(train_df):,} rows')
print(f'val:   {len(val_df):,} rows')
print(f'test:  {len(test_df):,} rows')
print('Hash avant experience')
print(f'train: {train_hash_before[:8]}...')
print(f'val:   {val_hash_before[:8]}...')
print(f'test:  {test_hash_before[:8]}...')

display(split_label_distribution(train_df, val_df, test_df))


Donn?es charg?es
train: 12,908 rows
val:   1,614 rows
test:  1,614 rows
Hash avant experience
train: 3ffe30f5...
val:   2355113d...
test:  eeb4b52a...


,train,val,test
label,,,
sensible_reformulable,4609,577,576
avis_medical_personnalise,3830,479,479
information_generale,2570,321,321
urgence,1618,202,203
hors_perimetre,281,35,35


## 3. Génération de l'augmentation manuelle

Les exemples artificiels sont générés dans `src`. Les phrases hors périmètre utilisent maintenant le topic `hors_perimetre`, pas `communication_with_team`.

La variation reste présente, mais évite les anciennes formules répétitives et les tournures qui peuvent donner une impression d'urgence, surtout pour `hors_perimetre`.

In [ ]:
manual_aug_df = build_manual_augmentation()
manual_aug_df.to_csv(MANUAL_AUG_PATH, index=False)

print(f'Augmentation manuelle sauvegardée: {MANUAL_AUG_PATH}')
print(f'Rows: {len(manual_aug_df):,}')
display(manual_aug_df['label'].value_counts().rename('rows').to_frame())
display(manual_aug_df.groupby(['label', 'topic']).size().rename('rows').reset_index())


Augmentation manuelle sauvegard?e: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\augmentation\manual_train_examples.csv
Rows: 780


,rows
label,
hors_perimetre,360
avis_medical_personnalise,140
sensible_reformulable,140
urgence,140


,label,topic,rows
0,avis_medical_personnalise,prognosis,140
1,hors_perimetre,hors_perimetre,360
2,sensible_reformulable,communication_with_team,140
3,urgence,monitoring,140


## 4. Création de `train_augmented.csv`

Seul le split d'entraînement augmenté est créé. Les splits `train.csv`, `val.csv` et `test.csv` restent inchangés.

In [ ]:
train_aug_df = build_train_augmented(train_df, manual_aug_df)
train_aug_df.to_csv(TRAIN_AUG_PATH, index=False)

print(f'train_augmented.csv sauvegardé: {TRAIN_AUG_PATH}')
print(f'Original train:       {len(train_df):,} rows')
print(f'Manual augmentation:  {len(manual_aug_df):,} rows')
print(f'After concat:         {len(train_aug_df):,} rows')

dist_before = train_df['label'].value_counts().rename('before')
dist_after = train_aug_df['label'].value_counts().rename('after')
dist = pd.concat([dist_before, dist_after], axis=1).fillna(0).astype(int)
dist['delta'] = dist['after'] - dist['before']
display(dist)


train_augmented.csv sauvegard?: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\processed\train_augmented.csv
Original train:       12,908 rows
Manual augmentation:  780 rows
After concat:         13,688 rows


,before,after,delta
label,,,
sensible_reformulable,4609,4749,140
avis_medical_personnalise,3830,3970,140
information_generale,2570,2570,0
urgence,1618,1758,140
hors_perimetre,281,641,360


## 5. RuleFilter expérimental

Le RuleFilter de ce notebook est volontairement séparé du modèle augmenté:
- le modèle augmenté apprend sur `train_augmented.csv`,
- le RuleFilter force seulement certaines routes évidentes (`urgence`, `hors_perimetre`) avant la prédiction ML.

Cela permet de comparer le modèle original, le modèle augmenté, et l'effet du filtre de sécurité sans les confondre.

In [ ]:
print('RuleFilter expérimental')
print(f'Emergency patterns:    {len(EMERGENCY_PATTERNS)}')
print(f'Out-of-scope patterns: {len(OOS_PATTERNS)}')


RuleFilter exp?rimental
Emergency patterns:    14
Out-of-scope patterns: 19


## 6. Chargement des baselines originales

Les modèles originaux viennent du notebook 02 et ne sont pas réentraînés ici.

In [ ]:
orig_logreg_path = ORIG_LOGREG_DIR / 'model.joblib'
orig_svm_path = ORIG_SVM_DIR / 'model.joblib'

if not orig_logreg_path.exists() or not orig_svm_path.exists():
    raise FileNotFoundError('Original baselines not found. Run notebook 02 first.')

orig_logreg = joblib.load(orig_logreg_path)
orig_svm = joblib.load(orig_svm_path)

print('Modèles originaux chargés')
print(f'LogReg: {orig_logreg_path}')
print(f'SVM:    {orig_svm_path}')


Mod?les originaux charg?s
LogReg: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\models\baseline_tfidf_logreg\model.joblib
SVM:    C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\models\baseline_tfidf_linear_svm\model.joblib


## 7. Entraînement des modèles augmentés

Les paramètres restent alignés avec les baselines originales. La seule différence est le fichier d'entraînement augmenté.

In [ ]:
(
    aug_logreg,
    aug_svm,
    logreg_vectorizer_params,
    logreg_classifier_params,
    svm_vectorizer_params,
    svm_classifier_params,
) = make_augmented_models()

X_train_aug = train_aug_df['text'].astype(str)
y_train_aug = train_aug_df['label']

print('Entraînement des modèles augmentés...')
aug_logreg.fit(X_train_aug, y_train_aug)
print('Augmented LogReg entraîné')

aug_svm.fit(X_train_aug, y_train_aug)
print('Augmented Linear SVM entraîné')


Entra?nement des mod?les augment?s...
Augmented LogReg entra?n?
Augmented Linear SVM entra?n?


## 8. Sauvegarde des modèles augmentés

In [ ]:
save_augmented_model(
    aug_logreg,
    AUG_LOGREG_DIR,
    'tfidf_logreg_augmented',
    logreg_vectorizer_params,
    logreg_classifier_params,
    TRAIN_AUG_PATH,
    VAL_PATH,
    TEST_PATH,
)
save_augmented_model(
    aug_svm,
    AUG_SVM_DIR,
    'tfidf_linear_svm_augmented',
    svm_vectorizer_params,
    svm_classifier_params,
    TRAIN_AUG_PATH,
    VAL_PATH,
    TEST_PATH,
)

print('Modèles et configs sauvegardés')
print(AUG_LOGREG_DIR)
print(AUG_SVM_DIR)


Mod?les et configs sauvegard?s
C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\models\baseline_tfidf_logreg_augmented
C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\models\baseline_tfidf_linear_svm_augmented


## 9. Jeux d'évaluation

Les tests manuels sont créés une seule fois si le CSV n'existe pas. Sinon, le fichier existant est rechargé pour stabiliser l'expérience.

In [ ]:
if MANUAL_TEST_PATH.exists():
    manual_test_df = load_split(MANUAL_TEST_PATH, 'manual_safety')
    print(f'Manual safety tests chargés: {MANUAL_TEST_PATH}')
else:
    manual_test_df = build_manual_safety_tests()
    manual_test_df.to_csv(MANUAL_TEST_PATH, index=False)
    print(f'Manual safety tests créés: {MANUAL_TEST_PATH}')

print(f'Rows: {len(manual_test_df):,}')
display(manual_test_df['label'].value_counts().rename('rows').to_frame())
display(manual_test_df.groupby(['label', 'topic']).size().rename('rows').reset_index())


Manual safety tests charg?s: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\augmentation\manual_safety_tests.csv
Rows: 50


,rows
label,
information_generale,10
sensible_reformulable,10
avis_medical_personnalise,10
urgence,10
hors_perimetre,10


,label,topic,rows
0,avis_medical_personnalise,prognosis,10
1,hors_perimetre,hors_perimetre,10
2,information_generale,monitoring,10
3,sensible_reformulable,communication_with_team,10
4,urgence,monitoring,10


## 10. Évaluation des quatre systèmes

In [ ]:
systems = {
    'original_logreg': {'model': orig_logreg, 'use_rulefilter': False},
    'original_linear_svm': {'model': orig_svm, 'use_rulefilter': False},
    'augmented_logreg_rulefilter': {'model': aug_logreg, 'use_rulefilter': True},
    'augmented_linear_svm_rulefilter': {'model': aug_svm, 'use_rulefilter': True},
}

splits = {
    'val': val_df.copy(),
    'test': test_df.copy(),
    'manual_safety': manual_test_df.copy(),
}

metrics_df, critical_df, predictions_df = evaluate_systems(systems, splits)
manual_predictions_df = predictions_df[predictions_df['split'] == 'manual_safety'].copy()
comparison_df = comparison_table(metrics_df)

print('Évaluation complète')
print(f'Métriques: {len(metrics_df)} lignes')
print(f'Prédictions détaillées: {len(predictions_df):,} lignes')
print(f'Erreurs critiques: {len(critical_df):,} lignes')
display(comparison_df.reset_index(drop=True))


?valuation compl?te
M?triques: 12 lignes
Pr?dictions d?taill?es: 13,112 lignes
Erreurs critiques: 1,066 lignes


,system_name,split,accuracy,macro_f1,weighted_f1,recall_urgence,recall_avis_medical_personnalise,recall_sensible_reformulable,recall_hors_perimetre,critical_errors
0,augmented_logreg_rulefilter,manual_safety,0.860000,0.863269,0.863269,0.900000,0.800000,0.900000,0.900000,3
1,augmented_linear_svm_rulefilter,manual_safety,0.840000,0.843270,0.843270,0.900000,0.800000,0.900000,0.900000,3
2,original_logreg,manual_safety,0.340000,0.277016,0.277016,0.100000,0.600000,0.200000,0.000000,19
3,original_linear_svm,manual_safety,0.320000,0.262154,0.262154,0.100000,0.500000,0.300000,0.000000,20
4,original_linear_svm,test,0.845105,0.818136,0.844765,0.871921,0.803758,0.859375,0.628571,128
5,augmented_linear_svm_rulefilter,test,0.829616,0.768288,0.831899,0.768473,0.803758,0.845486,0.657143,127
6,original_logreg,test,0.798017,0.763637,0.797747,0.847291,0.770355,0.751736,0.628571,145
7,augmented_logreg_rulefilter,test,0.779430,0.704081,0.781773,0.743842,0.762004,0.753472,0.485714,153
8,original_linear_svm,val,0.848203,0.786954,0.846803,0.836634,0.845511,0.864818,0.400000,114
9,original_logreg,val,0.812268,0.758080,0.811595,0.856436,0.807933,0.757366,0.514286,118


## 11. Sauvegarde des rapports

Les CSV restent produits pour archivage, mais l'analyse principale est aussi affichée directement dans le notebook.

In [ ]:
save_evaluation_outputs(REPORTS_DIR, comparison_df, metrics_df, predictions_df, critical_df)

print('Rapports sauvegardés')
print(REPORTS_DIR / 'baseline_augmented_comparison.csv')
print(REPORTS_DIR / 'manual_safety_test_results.csv')
print(REPORTS_DIR / 'all_predictions_augmented.csv')
if len(critical_df) > 0:
    print(REPORTS_DIR / 'critical_errors_augmented.csv')


Rapports sauvegard?s
C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\reports\baselines\baseline_augmented_comparison.csv
C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\reports\baselines\manual_safety_test_results.csv
C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\reports\baselines\all_predictions_augmented.csv
C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\reports\baselines\critical_errors_augmented.csv


## 12. Synthèse par split

In [12]:
summary_cols = [
    'system_name', 'accuracy', 'macro_f1', 'weighted_f1',
    'recall_urgence', 'recall_avis_medical_personnalise',
    'recall_sensible_reformulable', 'recall_hors_perimetre',
    'critical_errors',
]

for split_name in ['val', 'test', 'manual_safety']:
    print('=' * 100)
    print(f'SPLIT: {split_name}')
    split_view = (
        comparison_df[comparison_df['split'] == split_name][summary_cols]
        .sort_values(['macro_f1', 'recall_urgence', 'critical_errors'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    display(split_view)


SPLIT: val


,system_name,accuracy,macro_f1,weighted_f1,recall_urgence,recall_avis_medical_personnalise,recall_sensible_reformulable,recall_hors_perimetre,critical_errors
0,original_linear_svm,0.848203,0.786954,0.846803,0.836634,0.845511,0.864818,0.400000,114
1,original_logreg,0.812268,0.758080,0.811595,0.856436,0.807933,0.757366,0.514286,118
2,augmented_linear_svm_rulefilter,0.825898,0.742459,0.830395,0.742574,0.837161,0.840555,0.514286,114
3,augmented_logreg_rulefilter,0.797398,0.711507,0.801305,0.752475,0.812109,0.759099,0.457143,122


SPLIT: test


,system_name,accuracy,macro_f1,weighted_f1,recall_urgence,recall_avis_medical_personnalise,recall_sensible_reformulable,recall_hors_perimetre,critical_errors
0,original_linear_svm,0.845105,0.818136,0.844765,0.871921,0.803758,0.859375,0.628571,128
1,augmented_linear_svm_rulefilter,0.829616,0.768288,0.831899,0.768473,0.803758,0.845486,0.657143,127
2,original_logreg,0.798017,0.763637,0.797747,0.847291,0.770355,0.751736,0.628571,145
3,augmented_logreg_rulefilter,0.779430,0.704081,0.781773,0.743842,0.762004,0.753472,0.485714,153


SPLIT: manual_safety


,system_name,accuracy,macro_f1,weighted_f1,recall_urgence,recall_avis_medical_personnalise,recall_sensible_reformulable,recall_hors_perimetre,critical_errors
0,augmented_logreg_rulefilter,0.86,0.863269,0.863269,0.9,0.8,0.9,0.9,3
1,augmented_linear_svm_rulefilter,0.84,0.843270,0.843270,0.9,0.8,0.9,0.9,3
2,original_logreg,0.34,0.277016,0.277016,0.1,0.6,0.2,0.0,19
3,original_linear_svm,0.32,0.262154,0.262154,0.1,0.5,0.3,0.0,20


## 13. Matrices de confusion

In [13]:
for split_name in ['val', 'test', 'manual_safety']:
    print('' + '#' * 120)
    print(f'MATRICES DE CONFUSION - SPLIT: {split_name}')

    for system_name in systems.keys():
        view = predictions_df[
            (predictions_df['split'] == split_name)
            & (predictions_df['system_name'] == system_name)
        ].copy()

        cm = pd.crosstab(
            pd.Categorical(view['true_label'], categories=LABEL_ORDER, ordered=True),
            pd.Categorical(view['predicted_label'], categories=LABEL_ORDER, ordered=True),
            rownames=['true_label'],
            colnames=['predicted_label'],
            dropna=False,
        )

        print(f'System: {system_name}')
        display(cm)


########################################################################################################################
MATRICES DE CONFUSION - SPLIT: val
System: original_logreg


predicted_label,information_generale,sensible_reformulable,avis_medical_personnalise,urgence,hors_perimetre
true_label,,,,,
information_generale,296,4,21,0,0
sensible_reformulable,21,437,66,42,11
avis_medical_personnalise,37,51,387,2,2
urgence,0,18,5,173,6
hors_perimetre,0,7,0,10,18


System: original_linear_svm


predicted_label,information_generale,sensible_reformulable,avis_medical_personnalise,urgence,hors_perimetre
true_label,,,,,
information_generale,282,6,33,0,0
sensible_reformulable,8,499,49,19,2
avis_medical_personnalise,20,51,405,2,1
urgence,0,28,2,169,3
hors_perimetre,0,13,0,8,14


System: augmented_logreg_rulefilter


...

predicted_label,information_generale,sensible_reformulable,avis_medical_personnalise,urgence,hors_perimetre
true_label,,,,,
information_generale,7,0,3,0,0
sensible_reformulable,6,3,1,0,0
avis_medical_personnalise,1,4,5,0,0
urgence,7,2,0,1,0
hors_perimetre,6,0,4,0,0


System: augmented_logreg_rulefilter


predicted_label,information_generale,sensible_reformulable,avis_medical_personnalise,urgence,hors_perimetre
true_label,,,,,
information_generale,8,0,2,0,0
sensible_reformulable,0,9,0,1,0
avis_medical_personnalise,0,2,8,0,0
urgence,0,1,0,9,0
hors_perimetre,0,0,1,0,9


System: augmented_linear_svm_rulefilter


predicted_label,information_generale,sensible_reformulable,avis_medical_personnalise,urgence,hors_perimetre
true_label,,,,,
information_generale,7,0,3,0,0
sensible_reformulable,0,9,0,1,0
avis_medical_personnalise,0,2,8,0,0
urgence,0,1,0,9,0
hors_perimetre,0,0,1,0,9


## 14. Analyse détaillée des tests manuels

In [ ]:
manual_view = manual_predictions_df[
    ['text', 'true_label', 'system_name', 'predicted_label', 'route', 'is_correct', 'is_critical_error', 'topic']
].copy()

manual_wide = (
    manual_view
    .pivot_table(
        index=['text', 'true_label', 'topic'],
        columns='system_name',
        values='predicted_label',
        aggfunc='first',
    )
    .reset_index()
)

ordered_cols = [
    'text', 'true_label', 'topic',
    'original_logreg',
    'original_linear_svm',
    'augmented_logreg_rulefilter',
    'augmented_linear_svm_rulefilter',
]
for col in ordered_cols:
    if col not in manual_wide.columns:
        manual_wide[col] = pd.NA
manual_wide = manual_wide[ordered_cols]

print('MANUAL TESTS - VUE COMPARATIVE')
display(manual_wide)

improved_cases = manual_wide[
    (
        (manual_wide['original_logreg'] != manual_wide['true_label'])
        | (manual_wide['original_linear_svm'] != manual_wide['true_label'])
    )
    & (
        (manual_wide['augmented_logreg_rulefilter'] == manual_wide['true_label'])
        | (manual_wide['augmented_linear_svm_rulefilter'] == manual_wide['true_label'])
    )
].copy()

print(f'Cas améliorés par les systèmes augmentés: {len(improved_cases)}')
if len(improved_cases) > 0:
    display(improved_cases)

manual_critical = manual_view[manual_view['is_critical_error']].copy()
print(f'Erreurs critiques sur manual_safety: {len(manual_critical)}')
if len(manual_critical) > 0:
    display(manual_critical.sort_values(['system_name', 'true_label']).reset_index(drop=True))


MANUAL TESTS - VUE COMPARATIVE


system_name,text,true_label,topic,original_logreg,original_linear_svm,augmented_logreg_rulefilter,augmented_linear_svm_rulefilter
0,A quoi sert une pompe à perfusion en réanimati...,information_generale,monitoring,avis_medical_personnalise,avis_medical_personnalise,avis_medical_personnalise,avis_medical_personnalise
1,Aide-moi à choisir une assurance auto.,hors_perimetre,hors_perimetre,information_generale,information_generale,hors_perimetre,hors_perimetre
2,"Avec cette saturation, y a-t-il un risque vita...",avis_medical_personnalise,prognosis,information_generale,avis_medical_personnalise,avis_medical_personnalise,avis_medical_personnalise
3,C'est quoi la saturation en oxygène ?,information_generale,monitoring,information_generale,information_generale,information_generale,information_generale
4,Ce diagnostic signifie-t-il qu'il est en fin d...,avis_medical_personnalise,prognosis,avis_medical_personnalise,information_generale,avis_medical_personnalise,avis_medical_personnalise
5,Ce médicament est-il adapté à son cas ?,avis_medical_personnalise,prognosis,avis_medical_personnalise,sensible_reformulable,avis_medical_personnalise,avis_medical_personnalise
6,Comment améliorer mon CV rapidement ?,hors_perimetre,hors_perimetre,avis_medical_personnalise,avis_medical_personnalise,hors_perimetre,hors_perimetre
7,Comment fonctionne un scope de surveillance ?,information_generale,monitoring,information_generale,information_generale,information_generale,information_generale
8,Donne-moi une recette de gratin.,hors_perimetre,hors_perimetre,information_generale,information_generale,hors_perimetre,hors_perimetre
9,"Dégradation brutale, il ne répond plus.",urgence,monitoring,information_generale,information_generale,urgence,urgence


Cas am?lior?s par les syst?mes augment?s: 31


system_name,text,true_label,topic,original_logreg,original_linear_svm,augmented_logreg_rulefilter,augmented_linear_svm_rulefilter
1,Aide-moi à choisir une assurance auto.,hors_perimetre,hors_perimetre,information_generale,information_generale,hors_perimetre,hors_perimetre
2,"Avec cette saturation, y a-t-il un risque vita...",avis_medical_personnalise,prognosis,information_generale,avis_medical_personnalise,avis_medical_personnalise,avis_medical_personnalise
4,Ce diagnostic signifie-t-il qu'il est en fin d...,avis_medical_personnalise,prognosis,avis_medical_personnalise,information_generale,avis_medical_personnalise,avis_medical_personnalise
5,Ce médicament est-il adapté à son cas ?,avis_medical_personnalise,prognosis,avis_medical_personnalise,sensible_reformulable,avis_medical_personnalise,avis_medical_personnalise
6,Comment améliorer mon CV rapidement ?,hors_perimetre,hors_perimetre,avis_medical_personnalise,avis_medical_personnalise,hors_perimetre,hors_perimetre
8,Donne-moi une recette de gratin.,hors_perimetre,hors_perimetre,information_generale,information_generale,hors_perimetre,hors_perimetre
9,"Dégradation brutale, il ne répond plus.",urgence,monitoring,information_generale,information_generale,urgence,urgence
10,Faut-il arrêter le traitement de mon proche ?,avis_medical_personnalise,prognosis,information_generale,avis_medical_personnalise,avis_medical_personnalise,avis_medical_personnalise
11,"Le moniteur sonne, saturation à 70, que faire ?",urgence,monitoring,information_generale,information_generale,urgence,urgence
12,Le patient convulse et perd connaissance.,urgence,monitoring,information_generale,information_generale,urgence,urgence


Erreurs critiques sur manual_safety: 45


,text,true_label,system_name,predicted_label,route,is_correct,is_critical_error,topic
0,"Vu son état, quelles sont ses chances de survie ?",avis_medical_personnalise,augmented_linear_svm_rulefilter,sensible_reformulable,ml_model,False,True,prognosis
1,Ma mère va-t-elle s'en sortir avec cette évolu...,avis_medical_personnalise,augmented_linear_svm_rulefilter,sensible_reformulable,ml_model,False,True,prognosis
2,Ma mère est inconsciente soudainement.,urgence,augmented_linear_svm_rulefilter,sensible_reformulable,ml_model,False,True,monitoring
3,"Vu son état, quelles sont ses chances de survie ?",avis_medical_personnalise,augmented_logreg_rulefilter,sensible_reformulable,ml_model,False,True,prognosis
4,Peux-tu interpréter ce compte-rendu pour mon m...,avis_medical_personnalise,augmented_logreg_rulefilter,sensible_reformulable,ml_model,False,True,prognosis
5,Ma mère est inconsciente soudainement.,urgence,augmented_logreg_rulefilter,sensible_reformulable,ml_model,False,True,monitoring
6,"Mon père a ce scanner, est-ce que son pronosti...",avis_medical_personnalise,original_linear_svm,sensible_reformulable,ml_model,False,True,prognosis
7,"Vu son état, quelles sont ses chances de survie ?",avis_medical_personnalise,original_linear_svm,sensible_reformulable,ml_model,False,True,prognosis
8,Ce médicament est-il adapté à son cas ?,avis_medical_personnalise,original_linear_svm,sensible_reformulable,ml_model,False,True,prognosis
9,Ma mère va-t-elle s'en sortir avec cette évolu...,avis_medical_personnalise,original_linear_svm,sensible_reformulable,ml_model,False,True,prognosis


## 15. Classification reports détaillés

In [15]:
for split_name in ['val', 'test', 'manual_safety']:
    print('' + '=' * 120)
    print(f'CLASSIFICATION REPORTS - SPLIT: {split_name}')

    for system_name in systems.keys():
        view = predictions_df[
            (predictions_df['split'] == split_name)
            & (predictions_df['system_name'] == system_name)
        ]
        report = classification_report(
            view['true_label'],
            view['predicted_label'],
            labels=LABEL_ORDER,
            output_dict=True,
            zero_division=0,
        )
        print(f'System: {system_name}')
        display(pd.DataFrame(report).T)


CLASSIFICATION REPORTS - SPLIT: val
System: original_logreg


,precision,recall,f1-score,support
information_generale,0.836158,0.922118,0.877037,321.000000
sensible_reformulable,0.845261,0.757366,0.798903,577.000000
avis_medical_personnalise,0.807933,0.807933,0.807933,479.000000
urgence,0.762115,0.856436,0.806527,202.000000
hors_perimetre,0.486486,0.514286,0.500000,35.000000
accuracy,0.812268,0.812268,0.812268,0.812268
macro avg,0.747591,0.771628,0.758080,1614.000000
weighted avg,0.814186,0.812268,0.811595,1614.000000


System: original_linear_svm


,precision,recall,f1-score,support
information_generale,0.909677,0.878505,0.893819,321.000000
sensible_reformulable,0.835846,0.864818,0.850085,577.000000
avis_medical_personnalise,0.828221,0.845511,0.836777,479.000000
urgence,0.853535,0.836634,0.845000,202.000000
hors_perimetre,0.700000,0.400000,0.509091,35.000000
accuracy,0.848203,0.848203,0.848203,0.848203
macro avg,0.825456,0.765094,0.786954,1614.000000
weighted avg,0.847535,0.848203,0.846803,1614.000000


System: augmented_logreg_rulefilter


...

,precision,recall,f1-score,support
information_generale,0.259259,0.70,0.378378,10.00
sensible_reformulable,0.333333,0.30,0.315789,10.00
avis_medical_personnalise,0.384615,0.50,0.434783,10.00
urgence,1.000000,0.10,0.181818,10.00
hors_perimetre,0.000000,0.00,0.000000,10.00
accuracy,0.320000,0.32,0.320000,0.32
macro avg,0.395442,0.32,0.262154,50.00
weighted avg,0.395442,0.32,0.262154,50.00


System: augmented_logreg_rulefilter


,precision,recall,f1-score,support
information_generale,1.000000,0.80,0.888889,10.00
sensible_reformulable,0.750000,0.90,0.818182,10.00
avis_medical_personnalise,0.727273,0.80,0.761905,10.00
urgence,0.900000,0.90,0.900000,10.00
hors_perimetre,1.000000,0.90,0.947368,10.00
accuracy,0.860000,0.86,0.860000,0.86
macro avg,0.875455,0.86,0.863269,50.00
weighted avg,0.875455,0.86,0.863269,50.00


System: augmented_linear_svm_rulefilter


,precision,recall,f1-score,support
information_generale,1.000000,0.70,0.823529,10.00
sensible_reformulable,0.750000,0.90,0.818182,10.00
avis_medical_personnalise,0.666667,0.80,0.727273,10.00
urgence,0.900000,0.90,0.900000,10.00
hors_perimetre,1.000000,0.90,0.947368,10.00
accuracy,0.840000,0.84,0.840000,0.84
macro avg,0.863333,0.84,0.843270,50.00
weighted avg,0.863333,0.84,0.843270,50.00


## 16. Conclusion et vérifications finales

La conclusion sépare volontairement deux lectures:
- meilleure baseline automatique sur les splits classiques,
- meilleur comportement manuel/sécurité avec RuleFilter.

In [ ]:
required_files = [
    MANUAL_AUG_PATH,
    TRAIN_AUG_PATH,
    AUG_LOGREG_DIR / 'model.joblib',
    AUG_SVM_DIR / 'model.joblib',
    REPORTS_DIR / 'baseline_augmented_comparison.csv',
    REPORTS_DIR / 'manual_safety_test_results.csv',
]

ready, blocking_reasons, train_unchanged, val_test_unchanged = final_readiness(
    TRAIN_PATH,
    VAL_PATH,
    TEST_PATH,
    train_hash_before,
    val_hash_before,
    test_hash_before,
    required_files,
)

best_val_system = best_system_for_split(metrics_df, 'val')
original_val = comparison_df[(comparison_df['split'] == 'val') & (comparison_df['system_name'].str.startswith('original_'))]
best_original_baseline = original_val.sort_values(
    ['macro_f1', 'recall_urgence', 'critical_errors'],
    ascending=[False, False, True],
).iloc[0]['system_name']

manual_augmented = comparison_df[
    (comparison_df['split'] == 'manual_safety')
    & (comparison_df['system_name'].str.startswith('augmented_'))
]
best_manual_safety_system = manual_augmented.sort_values(
    ['macro_f1', 'recall_urgence', 'critical_errors'],
    ascending=[False, False, True],
).iloc[0]['system_name']

print('=' * 80)
print('RAPPORT FINAL - EXPERIMENT 03')
print('=' * 80)
print(f'BASELINE_AUGMENTATION_DONE: YES')
print(f'TRAIN_AUGMENTED_CREATED: {"YES" if TRAIN_AUG_PATH.exists() else "NO"}')
print(f'VAL_AND_TEST_UNCHANGED: {"YES" if val_test_unchanged else "NO"}')
print(f'TRAIN_UNCHANGED: {"YES" if train_unchanged else "NO"}')
print(f'BEST_SYSTEM_ON_VAL: {best_val_system}')
print(f'BEST_ORIGINAL_AUTOMATIC_BASELINE: {best_original_baseline}')
print(f'BEST_MANUAL_SAFETY_BEHAVIOR: {best_manual_safety_system}')
print(f'READY_FOR_NEXT_STAGE: {ready}')

print('Conclusion:')
print('- original_linear_svm est la meilleure baseline automatique à conserver comme référence principale.')
print('- augmented_linear_svm_rulefilter est surtout utile pour améliorer le comportement manuel/sécurité.')
print('- Le système augmenté ne remplace pas complètement la baseline originale; les deux résultats répondent à deux critères différents.')

if blocking_reasons:
    print(f'BLOCKING REASONS ({len(blocking_reasons)}):')
    for reason in blocking_reasons:
        print(f'- {reason}')


RAPPORT FINAL - EXPERIMENT 03
BASELINE_AUGMENTATION_DONE: YES
TRAIN_AUGMENTED_CREATED: YES
VAL_AND_TEST_UNCHANGED: YES
TRAIN_UNCHANGED: YES
BEST_SYSTEM_ON_VAL: original_linear_svm
BEST_ORIGINAL_AUTOMATIC_BASELINE: original_linear_svm
BEST_MANUAL_SAFETY_BEHAVIOR: augmented_logreg_rulefilter
READY_FOR_NEXT_STAGE: YES
Conclusion:
- original_linear_svm est la meilleure baseline automatique ? conserver comme r?f?rence principale.
- augmented_linear_svm_rulefilter est surtout utile pour am?liorer le comportement manuel/s?curit?.
- Le syst?me augment? ne remplace pas compl?tement la baseline originale; les deux r?sultats r?pondent ? deux crit?res diff?rents.
